# 21B — PPO with Tiny GPT

## Goal

Build a minimal Transformer-based Actor-Critic system for
autoregressive reinforcement learning.

We reuse the PPO mathematics and rollout contracts developed
in Lesson 21A, rather than implementing those algorithms again.

Our learning protocol is:

1. Understand the architecture and tensor contracts.
2. Verify causal action and state-value alignment.
3. Collect real autoregressive trajectories.
4. Train the Actor and Critic using PPO.
5. Evaluate actual task performance.
6. Extract only verified, reusable components.

The first experiment uses independently parameterized
Transformer Actor and Critic models.

In [1]:
# Import the models and reusable PPO components used in this notebook.

import torch
from torch import nn as nn

from llmfp.models import GPT, GPTConfig
from llmfp.rl import (
    RolloutBatch,
    align_state_values_for_actions,
    generalized_advantage_estimation,
    ppo_policy_loss,
    ppo_value_loss,
    rescore_actions_from_logits,
    select_rollout_sequences,
)

In [2]:
torch.manual_seed(42)

config = GPTConfig(
    vocab_size=5,
    embedding_dim=32,
    num_query_heads=4,
    num_kv_heads=2,
    num_layers=2,
    hidden_dim=64,
)

actor = GPT(config)

In [3]:
from typing import Any


class TinyGPTCritic(nn.Module):
    """Estimate causal state values using a Transformer and a value head."""

    def __init__(self, config: GPTConfig) -> None:
        super().__init__()

        # Reuse the exsiting GPT components, excluding its LM head.
        backbone = GPT(config)

        self.token_embedding = backbone.token_embedding
        self.blocks = backbone.blocks
        self.final_norm = backbone.final_norm

        self.value_head = nn.Linear(config.embedding_dim, 1)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        """Return a scalar state value per token position, shape (B, L)."""

        if token_ids.ndim != 2:
            raise ValueError("token_ids must have shape (B, L)")

        hidden_states = self.token_embedding(token_ids)

        for block in self.blocks:
            block_output = block(hidden_states)

            if not isinstance(block_output, torch.Tensor):
                raise RuntimeError(
                    "Expected a Transformer block output without a cache."
                )

            hidden_states = block_output

        hidden_states = self.final_norm(hidden_states)
        values = self.value_head(hidden_states)

        return values.squeeze(-1)

In [4]:
critic = TinyGPTCritic(config)
token_ids = torch.tensor([[0, 1, 3, 4], [0, 2, 4, 3]])

actor_logits = actor(token_ids)
critic_values = critic(token_ids)

print("Actor logits:", actor_logits.shape)
print("Critic values:", critic_values.shape)

assert actor_logits.shape == (2, 4, 5)
assert critic_values.shape == (2, 4)

Actor logits: torch.Size([2, 4, 5])
Critic values: torch.Size([2, 4])


In [5]:
action_token_positions = torch.tensor([[2, 3], [2, 3]])

action_mask = torch.ones((2, 2), dtype=torch.bool)

action_log_probs = rescore_actions_from_logits(
    actor_logits, token_ids, action_token_positions, action_mask
)

aligned_values = align_state_values_for_actions(
    critic_values, action_token_positions, action_mask
)

assert action_log_probs.shape == (2, 2)
assert aligned_values.shape == (2, 3)

print("Action log-probs:", action_log_probs.shape)
print("Aligned values:", aligned_values.shape)

Action log-probs: torch.Size([2, 2])
Aligned values: torch.Size([2, 3])


In [6]:
sequence_a = torch.tensor([[0, 1, 3, 4]])
sequence_b = torch.tensor([[0, 1, 4, 3]])

with torch.no_grad():
    actor_a = actor(sequence_a)
    actor_b = actor(sequence_b)

    critic_a = critic(sequence_a)
    critic_b = critic(sequence_b)

torch.testing.assert_close(
    actor_a[:, 1, :], actor_b[:, 1, :], atol=1e-5, rtol=1e-5
)

torch.testing.assert_close(critic_a[:, 1], critic_b[:, 1], atol=1e-5, rtol=1e-5)

print("Causal invariance passed.")

Causal invariance passed.


In [7]:
#  Create fixed PPO targets for testing gradient flow.

with torch.no_grad():
    old_logits = actor(token_ids)
    # (B, L, V): sequences, token positions, vocabulary logits

    old_log_probs = rescore_actions_from_logits(
        old_logits, token_ids, action_token_positions, action_mask
    )
    # (B, T): sequences, generated actions

    old_token_values = critic(token_ids)
    # (B, L): sequences, token positions

    old_state_values = align_state_values_for_actions(
        old_token_values, action_token_positions, action_mask
    )
    # (B, T+1): sequences, current states plus final next state


# Synthetic learning signals for the gradient experiment.
fixed_advantages = torch.tensor(
    [[1.0, 0.5], [-0.5, 1.0]],
    device=token_ids.device,
)
# (B, T): sequences, generated actions

value_offsets = torch.tensor(
    [[0.3, -0.2], [-0.1, 0.4]],
    device=token_ids.device,
)
# (B, T): sequences, desired adjustments to state values

fixed_value_targets = old_state_values[:, :-1] + value_offsets
# (B, T): sequences, action-state value targets

In [8]:
# Verify that PPO learning targets are frozen.

assert not old_log_probs.requires_grad
assert not old_state_values.requires_grad
assert not fixed_advantages.requires_grad
assert not fixed_value_targets.requires_grad

assert old_log_probs.shape == (2, 2)
assert old_state_values.shape == (2, 3)
assert fixed_advantages.shape == (2, 2)
assert fixed_value_targets.shape == (2, 2)

print("Frozen PPO targets verified.")

Frozen PPO targets verified.


In [9]:
# Recompute differentiable Actor and Critic predictions
current_logits = actor(
    token_ids
)  # (B, L, V): batch, sequence length, vocab logits

current_log_probs = rescore_actions_from_logits(
    current_logits, token_ids, action_token_positions, action_mask
)  # (B, T): batch, generated actions

current_token_values = critic(token_ids)  # (B, L): batch, seq len.

current_state_values = align_state_values_for_actions(
    current_token_values, action_token_positions, action_mask
)  # (B, T+1): batch, action states + next state

current_values = current_state_values[:, :-1]
# (B, T): sequences, action-state value estimates


policy_loss, approx_kl, clip_fraction = ppo_policy_loss(
    current_log_probabilities=current_log_probs,
    old_log_probabilities=old_log_probs,
    advantages=fixed_advantages,
    action_mask=action_mask,
    clip_epsilon=0.2,
)

value_loss = ppo_value_loss(
    current_values=current_values,
    value_targets=fixed_value_targets,
    action_mask=action_mask,
)

print("Policy loss:", policy_loss.item())
print("Value loss:", value_loss.item())
print("Approximate KL:", approx_kl.item())

Policy loss: -0.5
Value loss: 0.03750000149011612
Approximate KL: 0.0


In [10]:
#  Verify that PPO policy loss updates only the Actor gradient path.

actor.zero_grad(set_to_none=True)
critic.zero_grad(set_to_none=True)

# Both networks have independent parameters.
actor_parameter_ids = {id(p) for p in actor.parameters()}
critic_parameter_ids = {id(p) for p in critic.parameters()}

assert actor_parameter_ids.isdisjoint(critic_parameter_ids)

# Backpropagate only the PPO policy loss.
policy_loss.backward()

actor_embedding_grad = actor.token_embedding.weight.grad
# (V, C): vocabulary tokens, embedding features

assert actor_embedding_grad is not None
assert torch.isfinite(actor_embedding_grad).all()
assert actor_embedding_grad.norm().item() > 0.0

# Critic should not receive gradients from the policy loss.
assert all(parameter.grad is None for parameter in critic.parameters())

print("Actor embedding gradient norm:", actor_embedding_grad.norm().item())

print("Actor-only backward passed.")

Actor embedding gradient norm: 0.2249913066625595
Actor-only backward passed.


In [11]:
#  Verify that value loss changes only Critic gradients.

actor_grad_before_value = actor.token_embedding.weight.grad.detach().clone()
# (V, C): vocabulary tokens, Actor embedding features

value_loss.backward()

critic_embedding_grad = critic.token_embedding.weight.grad
# (V, C): vocabulary tokens, Critic embedding features

critic_head_grad = critic.value_head.weight.grad
# (1, C): one value output, embedding features

assert critic_embedding_grad is not None
assert critic_head_grad is not None

assert torch.isfinite(critic_embedding_grad).all()
assert torch.isfinite(critic_head_grad).all()

assert critic_head_grad.norm().item() > 0.0

# The Critic backward pass must not change Actor gradients.
torch.testing.assert_close(
    actor.token_embedding.weight.grad,
    actor_grad_before_value,
    atol=0.0,
    rtol=0.0,
)

print("Critic embedding gradient norm:", critic_embedding_grad.norm().item())

print("Critic value-head gradient norm:", critic_head_grad.norm().item())

print("Independent Actor/Critic backward passed.")

Critic embedding gradient norm: 0.074314184486866
Critic value-head gradient norm: 0.6543530821800232
Independent Actor/Critic backward passed.


In [12]:
#  Inspect gradients at the embeddings, attention, and output heads.

gradient_checks: dict[str, torch.Tensor | None] = {
    "Actor embedding": actor.token_embedding.weight.grad,
    "Actor attention Q": (
        actor.blocks[0].attention.query_projection.weight.grad
    ),
    "Actor LM head": actor.lm_head.weight.grad,
    "Critic embedding": critic.token_embedding.weight.grad,
    "Critic attention Q": (
        critic.blocks[0].attention.query_projection.weight.grad
    ),
    "Critic value head": critic.value_head.weight.grad,
}

for name, gradient in gradient_checks.items():
    assert gradient is not None
    assert torch.isfinite(gradient).all()

    print(f"{name}: {gradient.norm().item():.6f}")

Actor embedding: 0.224991
Actor attention Q: 0.048742
Actor LM head: 2.007092
Critic embedding: 0.074314
Critic attention Q: 0.043790
Critic value head: 0.654353


### Gradient Flow and Parameter Isolation — Takeaways

We connected the reusable PPO objectives to a Transformer-based
Actor-Critic architecture.

The experiment verifies three properties:

1. Collection-time log-probabilities and learning targets are
   detached from autograd.
2. The PPO policy loss propagates gradients through the Actor's
   language-model head, Transformer layers, and token embeddings.
3. The value loss propagates gradients through the Critic's
   value head, Transformer layers, and token embeddings.

The Actor and Critic have independent parameters and receive
gradients only from their corresponding objectives.

This experiment uses synthetic advantages and value targets to
test computational correctness. It is not yet a real RL rollout.

The next step is to collect autoregressive GPT trajectories,
compute GAE targets, and perform a genuine PPO update.

In [13]:
#  Evaluate complete two-token responses using deterministic rules.


def score_toy_completions(
    prompt_kinds: torch.Tensor,
    completion_ids: torch.Tensor,
) -> torch.Tensor:
    """Return one sequence-level reward for each completed response.

    Args:
        prompt_kinds: (B,) — prompt type for each sequence.
        completion_ids: (B, T=2) — two generated token IDs per sequence.

    Returns:
        (B,) — one scalar reward per completed sequence.
    """
    batch_size = prompt_kinds.shape[0]

    if prompt_kinds.shape != (batch_size,):
        raise ValueError("prompt_kinds must have shape (B,).")

    if completion_ids.shape != (batch_size, 2):
        raise ValueError("completion_ids must have shape (B, 2).")

    # (B,): expected token ID at each generated position.
    target_first = torch.where(prompt_kinds == 1, 3, 4)
    target_second = torch.where(prompt_kinds == 1, 4, 3)

    # (B,): correctness of each action.
    first_correct = completion_ids[:, 0] == target_first
    second_correct = completion_ids[:, 1] == target_second

    # (B,): exact correctness of the entire completion.
    both_correct = first_correct & second_correct

    # (B,): terminal sequence-level rewards.
    sequence_rewards = (
        0.25 * first_correct.float()
        + 0.25 * second_correct.float()
        + 0.5 * both_correct.float()
    )

    return sequence_rewards

In [14]:
# Verify exact, partial, and zero-reward cases.

test_prompt_kinds = torch.tensor([1, 2, 1, 2])
# (B=4,): prompt type for each test sequence

test_completions = torch.tensor([[3, 4], [4, 3], [3, 3], [0, 0]])
# (B=4, T=2): sequences, generated action positions

test_rewards = score_toy_completions(
    test_prompt_kinds,
    test_completions,
)
# (B=4,): reward for each complete sequence

torch.testing.assert_close(
    test_rewards,
    torch.tensor([1.0, 1.0, 0.25, 0.0]),
)

print("Reward sanity test passed.")

Reward sanity test passed.


In [15]:
# Collect autoregressive GPT tracjectories and build frozen PPO targets.


@torch.no_grad()
def collect_gpt_toy_rollout(
    actor: GPT,
    critic: TinyGPTCritic,
    batch_size: int,
    gamma: float,
    gae_lambda: float,
) -> RolloutBatch:
    """Collect two-token GPT episodes for PPO training.

    Args:
        actor: Autoregressive Transformer policy.
        critic: Transformer state-value estimator.
        batch_size: Number of independent trajectories, B.
        gamma: Reward discount factor.
        gae_lambda: GAE trace-decay parameter.

    Returns:
        RolloutBatch containing frozen collection facts and learning targets.
    """
    if batch_size <= 0:
        raise ValueError("batch size must be positive.")

    device = next(actor.parameters()).device

    if next(critic.parameters()).device != device:
        raise ValueError("Actor and Critic must use the same device.")

    # Save model modes so collection does not change caller state.
    actor_was_training = actor.training
    critic_was_training = critic.training

    actor.eval()
    critic.eval()

    try:
        # --------------------------------------------------
        # 1. Construct prompts.
        # --------------------------------------------------
        # (B,): prompt type, either 1 or 2.
        prompt_kinds = torch.randint(1, 3, (batch_size,), device=device)
        # (B, L_prompt=2):seqs, prompt token positions
        token_ids = torch.stack(
            [torch.zeros_like(prompt_kinds), prompt_kinds], dim=1
        )

        sampled_log_probs: list[torch.Tensor] = []

        # --------------------------------------------------
        # 2. Autoregressively sample two actions.
        # --------------------------------------------------
        for _ in range(2):
            # (B, V): seqs, next-token vacabulary logits.
            next_logits = actor(token_ids)[:, -1, :]
            # (B, V): log-prob for each possible token.
            next_log_probs = torch.log_softmax(next_logits, dim=-1)
            # (B, V): categorical sampling probs
            next_probabilities = torch.exp(next_log_probs)
            # (B, 1): one sampled action per seq.
            sampled_tokens = torch.multinomial(
                next_probabilities, num_samples=1
            )

            # (B,): seq indices for advanced indexing
            row_indices = torch.arange(batch_size, device=device)

            # (B,): log-prob of each sampled action.
            selected_log_probs = next_log_probs[
                row_indices, sampled_tokens.squeeze(1)
            ]
            sampled_log_probs.append(selected_log_probs)

            # The sampled action becomes part of the next state.
            # (B, L_current): prompt plus generated tokens so far
            token_ids = torch.cat([token_ids, sampled_tokens], dim=1)

        # (B, T=2): behavior-policy log-prob per action.
        old_log_probabilities = torch.stack(sampled_log_probs, dim=1)

        # --------------------------------------------------
        # 3. Construct action alignment metadata.
        # --------------------------------------------------
        # (B, T=2): token positions of generated actions
        action_token_positions = (
            torch.arange(2, 4, device=device)
            .unsqueeze(0)
            .expand(batch_size, -1)
        )

        # (B, T=2): True for every real action.
        action_mask = torch.ones(
            (batch_size, 2), dtype=torch.bool, device=device
        )

        # --------------------------------------------------
        # 4. Evaluate complete sequences.
        # --------------------------------------------------
        # (B,): one reward per completed response
        sequence_rewards = score_toy_completions(prompt_kinds, token_ids[:, 2:])
        # (B, T=2): transition rewards, initially zero
        rewards = torch.zeros(
            (batch_size, 2), dtype=sequence_rewards.dtype, device=device
        )

        # The environment returns rewards after the final action.
        rewards[:, -1] = sequence_rewards

        # --------------------------------------------------
        # 5. Construct episode boundary metadata.
        # --------------------------------------------------

        # Both (B, T): transition boundary switches
        bootstrap_mask = torch.ones_like(rewards)
        continuation_mask = torch.ones_like(rewards)

        # The second action ends the episode.
        bootstrap_mask[:, -1] = 0.0
        continuation_mask[:, -1] = 0.0

        # --------------------------------------------------
        # 6. Estimate Critic state values.
        # --------------------------------------------------

        # (B, L=4): one causal value prediction per token position
        token_values = critic(token_ids)

        # (B, T+1=3): V(s0), V(s1), V(s2)
        rollout_values = align_state_values_for_actions(
            token_values,
            action_token_positions,
            action_mask,
        )

        # --------------------------------------------------
        # 7. Compute fixed GAE learning targets.
        # --------------------------------------------------

        advantage_rows: list[torch.Tensor] = []
        target_rows: list[torch.Tensor] = []

        for index in range(batch_size):
            # Each (T,): advantages / value targets for one episode
            advantage_row, target_row = generalized_advantage_estimation(
                rewards=rewards[index],
                values=rollout_values[index],
                bootstrap_mask=bootstrap_mask[index],
                continuation_mask=continuation_mask[index],
                gamma=gamma,
                gae_lambda=gae_lambda,
            )

            advantage_rows.append(advantage_row)
            target_rows.append(target_row)

        # Both (B, T): fixed token-level PPO learning targets
        advantages = torch.stack(advantage_rows, dim=0)
        value_targets = torch.stack(target_rows, dim=0)

        # --------------------------------------------------
        # 8. Build the validated frozen rollout.
        # --------------------------------------------------

        rollout = RolloutBatch(
            token_ids=token_ids,
            action_token_positions=action_token_positions,
            action_mask=action_mask,
            old_log_probabilities=old_log_probabilities,
            rewards=rewards,
            rollout_values=rollout_values,
            bootstrap_mask=bootstrap_mask,
            continuation_mask=continuation_mask,
            advantages=advantages,
            value_targets=value_targets,
        )

    finally:
        actor.train(actor_was_training)
        critic.train(critic_was_training)

    return rollout


In [16]:
# Collect the first GPT-generated rollout without changing model parameters.

torch.manual_seed(42)

actor.zero_grad(set_to_none=True)
critic.zero_grad(set_to_none=True)

gpt_rollout = collect_gpt_toy_rollout(
    actor=actor, critic=critic, batch_size=32, gamma=0.9, gae_lambda=0.95
)

# (B, L): complete token sequences
print("Token IDs:", gpt_rollout.token_ids.shape)

# (B, T): sampled-action log-probabilities
print("Old log-probs:", gpt_rollout.old_log_probabilities.shape)

# (B, T+1): aligned Critic values
print("State values:", gpt_rollout.rollout_values.shape)

# (B, T): GAE advantages
print("Advantages:", gpt_rollout.advantages.shape)

Token IDs: torch.Size([32, 4])
Old log-probs: torch.Size([32, 2])
State values: torch.Size([32, 3])
Advantages: torch.Size([32, 2])


In [17]:
#  Inspect sampled completions and their terminal rewards.

for index in range(5):
    # (L_prompt=2,): prompt tokens for one trajectory
    prompt = gpt_rollout.token_ids[index, :2]

    # (T=2,): generated action tokens for one trajectory
    completion = gpt_rollout.token_ids[index, 2:]

    # (): scalar reward for the completed sequence
    terminal_reward = gpt_rollout.rewards[index, -1]

    print(
        f"Prompt: {prompt.tolist()} | "
        f"Completion: {completion.tolist()} | "
        f"Reward: {terminal_reward.item():.2f}"
    )

Prompt: [0, 1] | Completion: [4, 0] | Reward: 0.00
Prompt: [0, 2] | Completion: [3, 0] | Reward: 0.00
Prompt: [0, 1] | Completion: [1, 1] | Reward: 0.00
Prompt: [0, 1] | Completion: [0, 1] | Reward: 0.00
Prompt: [0, 1] | Completion: [1, 3] | Reward: 0.00


In [18]:
# Verify equivalence between autoregressive sampling and causal rescoring.

actor_was_training = actor.training
actor.eval()

try:
    with torch.no_grad():
        # (B, L, V): sequences, token positions, vocabulary logits
        full_logits = actor(gpt_rollout.token_ids)

        # (B, T): log-probabilities of sampled actions
        rescored_log_probs = rescore_actions_from_logits(
            full_logits,
            gpt_rollout.token_ids,
            gpt_rollout.action_token_positions,
            gpt_rollout.action_mask,
        )

        # (B, T): discrepancy between both evaluation paths
        log_prob_difference = (
            rescored_log_probs - gpt_rollout.old_log_probabilities
        )

        # (B, T): current / behavior-policy probability ratios
        probability_ratios = torch.exp(log_prob_difference)

finally:
    actor.train(actor_was_training)

torch.testing.assert_close(
    rescored_log_probs, gpt_rollout.old_log_probabilities, atol=1e-5, rtol=1e-5
)

torch.testing.assert_close(
    probability_ratios,
    torch.ones_like(probability_ratios),
    atol=1e-5,
    rtol=1e-5,
)

print(
    "Maximum log-probability difference:",
    log_prob_difference.abs().max().item(),
)

print("Autoregressive rescoring invariant passed.")

Maximum log-probability difference: 2.384185791015625e-07
Autoregressive rescoring invariant passed.


In [19]:
# Verify two-step GAE using stored GPT Critic values.

# (T+1=3,): V(s0), V(s1), V(s2) for one trajectory
values = gpt_rollout.rollout_values[0]

# (T=2,): immediate rewards for two actions
rewards = gpt_rollout.rewards[0]

# (): final TD error; terminal transition has no bootstrap
delta_1 = rewards[1] - values[1]

# (): first TD error; next-state bootstrap is allowed
delta_0 = rewards[0] + 0.9 * values[1] - values[0]

expected_advantage_1 = delta_1

expected_advantage_0 = delta_0 + 0.9 * 0.95 * expected_advantage_1

# (T=2,): manually reconstructed advantages
expected_advantages = torch.stack([expected_advantage_0, expected_advantage_1])

torch.testing.assert_close(
    gpt_rollout.advantages[0], expected_advantages, atol=1e-5, rtol=1e-5
)

torch.testing.assert_close(
    gpt_rollout.value_targets[0],
    expected_advantages + values[:-1],
    atol=1e-5,
    rtol=1e-5,
)

print("GAE consistency test passed.")

GAE consistency test passed.


In [20]:
# Verify tensor shapes, termination semantics, and frozen rollout data.

batch_size = gpt_rollout.token_ids.shape[0]

assert gpt_rollout.token_ids.shape == (batch_size, 4)
assert gpt_rollout.old_log_probabilities.shape == (batch_size, 2)
assert gpt_rollout.rollout_values.shape == (batch_size, 3)
assert gpt_rollout.advantages.shape == (batch_size, 2)
assert gpt_rollout.value_targets.shape == (batch_size, 2)

# Only the final action receives sequence-level reward.
assert torch.all(gpt_rollout.rewards[:, 0] == 0.0)

# Final action is a true terminal transition.
assert torch.all(gpt_rollout.bootstrap_mask[:, -1] == 0.0)
assert torch.all(gpt_rollout.continuation_mask[:, -1] == 0.0)

# Collection does not retain autograd graphs or create parameter gradients.
assert not gpt_rollout.old_log_probabilities.requires_grad
assert not gpt_rollout.advantages.requires_grad
assert not gpt_rollout.value_targets.requires_grad

assert all(parameter.grad is None for parameter in actor.parameters())
assert all(parameter.grad is None for parameter in critic.parameters())

print("GPT rollout collection checks passed.")

GPT rollout collection checks passed.


### Autoregressive GPT Rollout Collection — Takeaways

We collected complete token trajectories using a Transformer Actor
and evaluated them using a deterministic sequence-level reward.

The collection pipeline now performs:

1. Prompt construction.
2. Autoregressive token sampling under the current GPT policy.
3. Recording the log-probability of each sampled action.
4. Sequence-level reward evaluation.
5. Causal state-value estimation with a Transformer Critic.
6. GAE computation using explicit episode-boundary masks.
7. Construction of a validated, frozen `RolloutBatch`.

We verified two important properties:

**Sampling–rescoring consistency**

Autoregressive generation and teacher-forced rescoring produce
matching action log-probabilities under the same unchanged policy.

**GAE consistency**

Terminal sequence rewards and Critic estimates produce the
expected token-level advantages without bootstrapping across
the final episode boundary.

Rollout collection runs without autograd. The stored trajectories
will be reused by PPO, while the current Actor and Critic
predictions will be recomputed during optimization.

The next step is to update the GPT Actor and Transformer Critic
using this frozen rollout.

## 05. One Real PPO Update with Tiny GPT

Use a frozen GPT rollout to perform one Actor update and one Critic
update.

```mermaid
flowchart TD
    R["Frozen rollout"] --> M["Select complete sequences"]
    M --> A["Current Actor forward"]
    M --> C["Current Critic forward"]

    A --> P["Action log-probabilities"]
    C --> V["Action-state values"]

    P --> PL["PPO policy loss"]
    V --> VL["Value loss"]

    PL --> AO["Actor backward and step"]
    VL --> CO["Critic backward and step"]

    AO --> E["Rescore the same stored actions"]
    CO --> E
```

Actions, old log-probabilities, advantages, and value targets remain
fixed. Only current-model predictions and model parameters change.

In [21]:
# Select one sequence minibatch and save pre-update snapshots.

actor.eval()
critic.eval()

batch_size = gpt_rollout.token_ids.shape[0]
minibatch_size = min(16, batch_size)

torch.manual_seed(7)

# (M,): indices of complete trajectories selected from the rollout
sequence_indices = torch.randperm(
    batch_size, device=gpt_rollout.token_ids.device
)[:minibatch_size]

minibatch = select_rollout_sequences(gpt_rollout, sequence_indices)

assert minibatch.token_ids.shape == (minibatch_size, 4)
assert minibatch.action_mask.shape == (minibatch_size, 2)

# These snapshots must stay unchanged throughout the PPO update.
# Each value is (B, T): one frozen quantity per collected action
rollout_snapshots = {
    "old_log_probabilities": gpt_rollout.old_log_probabilities.clone(),
    "advantages": gpt_rollout.advantages.clone(),
    "value_targets": gpt_rollout.value_targets.clone(),
}

# (V, C): vocabulary entries, Actor embedding features
actor_embedding_before = actor.token_embedding.weight.detach().clone()

# (1, C): scalar value output, Critic hidden features
critic_head_before = critic.value_head.weight.detach().clone()

print("Selected sequences:", minibatch_size)
print("Valid actions:", minibatch.action_mask.sum().item())

Selected sequences: 16
Valid actions: 32


In [22]:
# Recompute differentiable predictions and construct both PPO losses.

# (M, L, V): sequences, token positions, vocabulary logits
current_logits = actor(minibatch.token_ids)

# GPT can also return a cache; this experiment expects logits only.
assert isinstance(current_logits, torch.Tensor)

# (M, T): current log-probability of each stored sampled action
current_log_probs = rescore_actions_from_logits(
    current_logits,
    minibatch.token_ids,
    minibatch.action_token_positions,
    minibatch.action_mask,
)

# (M, L): sequences, causal value prediction at each token position
current_token_values = critic(minibatch.token_ids)

# (M, T+1): action-state values plus the final next-state value
current_state_values = align_state_values_for_actions(
    current_token_values,
    minibatch.action_token_positions,
    minibatch.action_mask,
)

# (M, T): values of the states from which actions were sampled
current_values = current_state_values[:, :-1]

# Each (): scalar policy loss or diagnostic
policy_loss, kl_before, clip_fraction_before = ppo_policy_loss(
    current_log_probabilities=current_log_probs,
    old_log_probabilities=minibatch.old_log_probabilities,
    advantages=minibatch.advantages,
    action_mask=minibatch.action_mask,
    clip_epsilon=0.2,
)

# (): scalar Critic loss
value_loss = ppo_value_loss(
    current_values=current_values,
    value_targets=minibatch.value_targets,
    action_mask=minibatch.action_mask,
)

assert policy_loss.requires_grad
assert value_loss.requires_grad
assert torch.isfinite(policy_loss).item()
assert torch.isfinite(value_loss).item()

# Before the FIRST update, the current policy must match the behavior policy.
torch.testing.assert_close(
    current_log_probs.detach(),
    minibatch.old_log_probabilities,
    atol=1e-5,
    rtol=1e-5,
)

print("Policy loss:", policy_loss.item())
print("Value loss:", value_loss.item())
print("Approximate KL before update:", kl_before.item())

Policy loss: -0.4180520176887512
Value loss: 0.14465749263763428
Approximate KL before update: 0.0


In [23]:
# Create independent optimizers for the Actor and Critic.

actor_parameters = list(actor.parameters())
critic_parameters = list(critic.parameters())

# Separate optimizers must not own the same trainable parameters.
assert {id(p) for p in actor_parameters}.isdisjoint(
    {id(p) for p in critic_parameters}
)

actor_optimizer = torch.optim.AdamW(actor_parameters, lr=3e-4, weight_decay=0.0)

critic_optimizer = torch.optim.AdamW(
    critic_parameters, lr=1e-3, weight_decay=0.0
)

In [24]:
# Compute independent gradients and clip each model's total norm.

actor_optimizer.zero_grad(set_to_none=True)
critic_optimizer.zero_grad(set_to_none=True)

# Only Actor parameters belong to the policy-loss computation graph.
policy_loss.backward()

assert all(parameter.grad is None for parameter in critic_parameters)

# The Critic has its own independent computation graph.
value_loss.backward()

# (V, C): gradients of Actor token embeddings
actor_embedding_grad = actor.token_embedding.weight.grad

# (1, C): gradients of the Critic value-head weights
critic_head_grad = critic.value_head.weight.grad

assert actor_embedding_grad is not None
assert critic_head_grad is not None

# Stop before any optimizer step if a gradient norm is NaN or infinite.
# (): total Actor gradient norm BEFORE clipping
actor_grad_norm = torch.nn.utils.clip_grad_norm_(
    actor_parameters,
    max_norm=1.0,
    error_if_nonfinite=True,
)

# (): total Critic gradient norm BEFORE clipping
critic_grad_norm = torch.nn.utils.clip_grad_norm_(
    critic_parameters,
    max_norm=1.0,
    error_if_nonfinite=True,
)

print("Actor gradient norm before clipping:", actor_grad_norm.item())
print("Critic gradient norm before clipping:", critic_grad_norm.item())

Actor gradient norm before clipping: 0.5798537731170654
Critic gradient norm before clipping: 1.870390772819519


In [25]:
# Update both models once and verify that rollout targets stay frozen.

actor_optimizer.step()
critic_optimizer.step()

# Optimizing model parameters must not mutate the original rollout.
for name, snapshot in rollout_snapshots.items():
    assert torch.equal(
        getattr(gpt_rollout, name),
        snapshot,
    ), f"Frozen rollout field changed: {name}"

# (): largest absolute change in an Actor embedding weight
actor_embedding_change = (
    (actor.token_embedding.weight.detach() - actor_embedding_before).abs().max()
)

# (): largest absolute change in a Critic value-head weight
critic_head_change = (
    (critic.value_head.weight.detach() - critic_head_before).abs().max()
)

print("Maximum Actor embedding change:", actor_embedding_change.item())
print("Maximum Critic head change:", critic_head_change.item())
print("Frozen rollout checks passed.")

Maximum Actor embedding change: 0.0003000497817993164
Maximum Critic head change: 0.0010000020265579224
Frozen rollout checks passed.


In [26]:
# Rescore the same minibatch and compare pre-update and post-update metrics.

with torch.no_grad():
    # (M, L, V): updated vocabulary logits
    updated_logits = actor(minibatch.token_ids)

    assert isinstance(updated_logits, torch.Tensor)

    # (M, T): updated log-probabilities of the SAME stored actions
    updated_log_probs = rescore_actions_from_logits(
        updated_logits,
        minibatch.token_ids,
        minibatch.action_token_positions,
        minibatch.action_mask,
    )

    # (M, T): updated policy probability / behavior-policy probability
    updated_ratios = torch.exp(
        updated_log_probs - minibatch.old_log_probabilities
    )

    # Each (): scalar post-update loss or diagnostic
    policy_loss_after, kl_after, clip_fraction_after = ppo_policy_loss(
        current_log_probabilities=updated_log_probs,
        old_log_probabilities=minibatch.old_log_probabilities,
        advantages=minibatch.advantages,
        action_mask=minibatch.action_mask,
        clip_epsilon=0.2,
    )

    # (M, L): updated causal Critic predictions
    updated_token_values = critic(minibatch.token_ids)

    # (M, T): updated action-state values
    updated_values = align_state_values_for_actions(
        updated_token_values,
        minibatch.action_token_positions,
        minibatch.action_mask,
    )[:, :-1]

    # (): scalar loss against the SAME fixed value targets
    value_loss_after = ppo_value_loss(
        updated_values, minibatch.value_targets, minibatch.action_mask
    )

# (N_valid,): ratios for valid sampled actions, excluding padding
valid_ratios = updated_ratios[minibatch.action_mask]

print(f"{'Metric':<20} {'Before':>12} {'After':>12}")

for name, before, after in (
    ("Policy loss", policy_loss, policy_loss_after),
    ("Value loss", value_loss, value_loss_after),
    ("Approximate KL", kl_before, kl_after),
    ("Clip fraction", clip_fraction_before, clip_fraction_after),
):
    print(f"{name:<20} {before.item():>12.6f} {after.item():>12.6f}")

print(
    f"Updated ratio range: "
    f"[{valid_ratios.min().item():.6f}, "
    f"{valid_ratios.max().item():.6f}]"
)

Metric                     Before        After
Policy loss             -0.418052    -0.426122
Value loss               0.144657     0.086870
Approximate KL           0.000000     0.000275
Clip fraction            0.000000     0.000000
Updated ratio range: [0.982793, 1.069763]


### One Real PPO Update — Takeaways

We used a genuinely sampled GPT rollout to update a Transformer Actor
and an independently parameterized Transformer Critic.

During this update:

- Sampled actions, old log-probabilities, advantages, and value targets
  remained fixed.
- Current predictions were recomputed with autograd enabled.
- The policy and value losses produced gradients for separate models.
- Gradient norms were clipped before the optimizer steps.
- The same stored actions were rescored after optimization.

`eval()` controls model behavior; it does not disable autograd.
Collection and diagnostic passes use `no_grad()`, while optimization
requires differentiable forward passes.

PPO ratio clipping and gradient-norm clipping operate at different
stages and constrain different quantities.

This experiment verifies one minibatch update. It does not yet
establish improved performance on newly generated responses.

## 06. Multiple PPO Epochs and Sequence Minibatches

A PPO update reuses one frozen rollout across multiple minibatches
and epochs.

The behavior-policy log-probabilities, advantages, and value targets
remain fixed for the entire update.

Before starting a new update cycle, collect fresh trajectories under
the latest policy.

For this experiment:

- Rollout batch size: 32 sequences
- Sequence minibatch size: 8
- PPO epochs: 3
- Expected optimizer steps: 12, unless early stopping is enabled

![PPO Inter-Outer Loop](./imgs/ppo_inter_outer_loop.png)

In [27]:
# Collect fresh trajectories under the updated GPT policy.

actor.eval()
critic.eval()

ppo_rollout = collect_gpt_toy_rollout(
    actor=actor,
    critic=critic,
    batch_size=32,
    gamma=0.9,
    gae_lambda=0.95,
)

# (B=32, L=4): prompt plus two generated tokens
assert ppo_rollout.token_ids.shape == (32, 4)

# (B=32, T=2): stored behavior-policy log-probabilities
assert ppo_rollout.old_log_probabilities.shape == (32, 2)

with torch.no_grad():
    # (B, L, V): current Actor logits before the new PPO update
    initial_logits = actor(ppo_rollout.token_ids)

    # (B, T): rescored log-probabilities of sampled actions
    initial_log_probs = rescore_actions_from_logits(
        initial_logits,
        ppo_rollout.token_ids,
        ppo_rollout.action_token_positions,
        ppo_rollout.action_mask,
    )

torch.testing.assert_close(
    initial_log_probs,
    ppo_rollout.old_log_probabilities,
    atol=1e-5,
    rtol=1e-5,
)

print("Fresh on-policy rollout verified.")

Fresh on-policy rollout verified.


In [28]:
# Resue one frozen rollout across multiple PPO epochs and minibatches


def update_gpt_ppo(
    actor: GPT,
    critic: TinyGPTCritic,
    rollout: RolloutBatch,
    actor_optimizer: torch.optim.Optimizer,
    critic_optimizer: torch.optim.Optimizer,
    *,
    num_epochs: int,
    minibatch_size: int,
    clip_epsilon: float,
    target_kl: float | None = None,
) -> dict[str, list[float]]:
    """Optimize the Actor and Critic against one fixed rollout.

    Args:
        actor: Current GPT token policy.
        critic: Current Transformer state-value estimator.
        rollout: Frozen trajectories and GAE learning targets.
        actor_optimizer: Optimizer for Actor parameters.
        critic_optimizer: Optimizer for Critic parameters.
        num_epochs: Number of passes over the rollout.
        minibatch_size: Number of complete sequences per minibatch.
        clip_epsilon: PPO probability-ratio clipping threshold.
        target_kl: Optional approximate-KL early stopping threshold.

    Returns:
        Per-update losses and full-rollout policy diagnostics.
    """
    if num_epochs < 1 or minibatch_size < 1:
        raise ValueError("num epochs and minibatch size must be positive.")
    if target_kl is not None and target_kl <= 0.0:
        raise ValueError("target kl must be positive.")
    if actor.training or critic.training:
        raise ValueError(
            "Use eval mode to match this lesson's rollout collection."
        )

    batch_size = rollout.token_ids.shape[0]
    history: dict[str, list[float]] = {
        "step": [],
        "epoch": [],
        "policy_loss": [],
        "value_loss": [],
        "approx_kl": [],
        "clip_fraction": [],
    }
    # --------------------------------------------------
    # 1. PPO epochs: reuse the same frozen rollout.
    # --------------------------------------------------
    for epoch in range(num_epochs):
        # (B,): shuffled indices of complete tracjectories
        sequence_order = torch.randperm(
            batch_size, device=rollout.token_ids.device
        )
        # --------------------------------------------------
        # 2. Sequence minibatches: update current models.
        # --------------------------------------------------
        for start in range(0, batch_size, minibatch_size):
            # (M_current,): indices selected for this minibatch
            selected_indices = sequence_order[start : start + minibatch_size]
            minibatch = select_rollout_sequences(rollout, selected_indices)
            # Recompute differentiable Actor perdiction.
            # (M_current, L, V): current vocabulary logits
            logits = actor(minibatch.token_ids)
            # (M_current, T): current log-probs of stored actions
            current_log_probs = rescore_actions_from_logits(
                logits,
                minibatch.token_ids,
                minibatch.action_token_positions,
                minibatch.action_mask,
            )

            # Recompute differentiable Critic predictions.
            # (M_current, L): causal values at token positions
            token_values = critic(minibatch.token_ids)

            # (M_current, T): values of action states only
            current_values = align_state_values_for_actions(
                token_values,
                minibatch.action_token_positions,
                minibatch.action_mask,
            )[:, :-1]

            # --------------------------------------------------
            # 3. Compute PPO objectives using fixed targets.
            # --------------------------------------------------
            policy_loss, _, _ = ppo_policy_loss(
                current_log_probabilities=current_log_probs,
                old_log_probabilities=minibatch.old_log_probabilities,
                advantages=minibatch.advantages,
                action_mask=minibatch.action_mask,
                clip_epsilon=clip_epsilon,
            )

            value_loss = ppo_value_loss(
                current_values=current_values,
                value_targets=minibatch.value_targets,
                action_mask=minibatch.action_mask,
            )

            # --------------------------------------------------
            # 4. Backpropagate and update independently.
            # --------------------------------------------------
            actor_optimizer.zero_grad(set_to_none=True)
            critic_optimizer.zero_grad(set_to_none=True)
            # Backpropagate each loss through its own computation graph.
            policy_loss.backward()
            value_loss.backward()

            # Prevent unsually large gradient steps.
            torch.nn.utils.clip_grad_norm_(
                actor.parameters(), max_norm=1.0, error_if_nonfinite=True
            )
            torch.nn.utils.clip_grad_norm_(
                critic.parameters(), max_norm=1.0, error_if_nonfinite=True
            )

            actor_optimizer.step()
            critic_optimizer.step()

            # ----------------------------------------------------
            # 5. Measure policy movement against the FULL rollout.
            # ----------------------------------------------------
            with torch.no_grad():
                # (B, L, V): updated Actor logits on all stored seqs
                full_logits = actor(rollout.token_ids)
                # (B, T): updated log-probs for all stored actions
                full_log_probs = rescore_actions_from_logits(
                    full_logits,
                    rollout.token_ids,
                    rollout.action_token_positions,
                    rollout.action_mask,
                )

                _, full_kl, full_clip_fraction = ppo_policy_loss(
                    current_log_probabilities=full_log_probs,
                    old_log_probabilities=rollout.old_log_probabilities,
                    advantages=rollout.advantages,
                    action_mask=rollout.action_mask,
                    clip_epsilon=clip_epsilon,
                )

            # Store diagnostics after this optimizer step.
            history["step"].append(float(len(history["step"]) + 1))
            history["epoch"].append(float(epoch + 1))
            history["policy_loss"].append(policy_loss.item())
            history["value_loss"].append(value_loss.item())
            history["approx_kl"].append(full_kl.item())
            history["clip_fraction"].append(full_clip_fraction.item())

            # Optionally stop reusing this rollout.
            if target_kl is not None and full_kl.item() > target_kl:
                return history

    return history

In [29]:
# Perform 12 optimizer steps while preserving one rollout snapshot.

torch.manual_seed(11)

# (B, T): frozen behavior-policy log-probabilities
old_snapshot = ppo_rollout.old_log_probabilities.clone()

# (B, T): fixed GAE advantages
advantage_snapshot = ppo_rollout.advantages.clone()

# (B, T): fixed Critic training targets
target_snapshot = ppo_rollout.value_targets.clone()

ppo_history = update_gpt_ppo(
    actor=actor,
    critic=critic,
    rollout=ppo_rollout,
    actor_optimizer=actor_optimizer,
    critic_optimizer=critic_optimizer,
    num_epochs=3,
    minibatch_size=8,
    clip_epsilon=0.2,
    target_kl=None,
)

# Three epochs, four minibatches per epoch.
assert len(ppo_history["step"]) == 12

# The stored rollout must remain unchanged throughout optimization.
assert torch.equal(ppo_rollout.old_log_probabilities, old_snapshot)

assert torch.equal(ppo_rollout.advantages, advantage_snapshot)

assert torch.equal(ppo_rollout.value_targets, target_snapshot)

print("Completed optimizer steps:", len(ppo_history["step"]))
print("Final approximate KL:", ppo_history["approx_kl"][-1])
print("Final clip fraction:", ppo_history["clip_fraction"][-1])
print("Frozen rollout checks passed.")

Completed optimizer steps: 12
Final approximate KL: 0.00670989416539669
Final clip fraction: 0.109375
Frozen rollout checks passed.


In [30]:
# Import interactive plotting for PPO diagnostics.
import plotly.graph_objects as go

# Visualize full-rollout policy movement across PPO updates.

figure = go.Figure()

figure.add_scatter(
    x=ppo_history["step"],
    y=ppo_history["approx_kl"],
    mode="lines+markers",
    name="Approximate KL",
)

figure.update_layout(
    title="PPO: Policy Movement During Rollout Reuse",
    xaxis_title="Optimizer step",
    yaxis_title="Approximate KL against behavior policy",
    hovermode="x unified",
)

figure.show()

### Multiple PPO Epochs — Takeaways

We extended one PPO optimizer step into a reusable multi-epoch,
sequence-minibatch training function.

For each PPO epoch, we shuffle complete trajectory indices
and iterate over sequence minibatches.

Each minibatch recomputes differentiable Actor and Critic predictions.

The same rollout-time actions, old log-probabilities, advantages,
and value targets remain fixed throughout the update cycle.

After each optimizer step, we measure approximate KL and clip
fraction against the original behavior policy using the full rollout.

An optional KL threshold can stop further rollout reuse when
the current policy has moved too far from the behavior policy.

This is a heuristic stopping condition, not a strict trust-region
constraint.

The next step is to collect a new rollout under the updated
Actor and repeat the process.